In [2]:
pip install -U langgraph

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install langgraph

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [9]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Set your Groq API key as an environment variable
os.environ["GROQ_API_KEY"] = "your_actual_api_key_here"

# 2. Now initialize your agent
agent = create_agent(
    model="groq:llama-3.3-70b-versatile", 
    checkpointer=InMemorySaver(), 
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("messages", 10),
            keep=("messages", 4)
        ),
    ]
)

In [10]:
config ={"configurable":{"thread_id":"test-1"}}

In [17]:
import os

os.environ["GROQ_API_KEY"] = "gsk_your_new_key"

print(os.environ["GROQ_API_KEY"][:10])

gsk_your_n


In [20]:
import os
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# Set your API key in the environment (or ensure it's exported in your terminal)
# os.environ["GROQ_API_KEY"] = "your_brand_new_key"

# 1. Initialize the model
model = init_chat_model(
    "groq:llama-3.3-70b-versatile"
)

# 2. Initialize the memory checkpointer
memory = MemorySaver()

# 3. Create the agent (passing the model and the memory saver)
# We pass an empty list for tools since this is just answering basic math
agent = create_react_agent(model, tools=[], checkpointer=memory)

# Define the config with a thread_id to track this specific conversation
config = {"configurable": {"thread_id": "thread-1"}}

questions = [
    "what is 2+2?",
    "what is 10-5?",
    "multiply the two answers together" # Added to test memory
]

# 4. Run the loop
for q in questions:
    print(f"User: {q}")
    
    # Invoke the agent
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    
    # Extract messages
    messages = response["messages"]
    
    print(f"Agent: {messages[-1].content}")
    print(f"Number of Messages in memory: {len(messages)}")
    print("-" * 50)

C:\Users\hp\AppData\Local\Temp\ipykernel_25476\3122849494.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools=[], checkpointer=memory)


User: what is 2+2?
Agent: 2 + 2 = 4.
Number of Messages in memory: 2
--------------------------------------------------
User: what is 10-5?
Agent: 10 - 5 = 5.
Number of Messages in memory: 4
--------------------------------------------------
User: multiply the two answers together
Agent: You want me to multiply the answers from the previous questions.

The answer to 2+2 was 4.
The answer to 10-5 was 5.

 Multiplying them together: 4 * 5 = 20.
Number of Messages in memory: 6
--------------------------------------------------


In [23]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("GROQ_API_KEY"))

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


In [25]:
import os
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langchain_core.messages import trim_messages
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# 1. Define the tool
@tool
def search_hotels(city: str) -> str:
    """Search hotels - return long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

# 2. Initialize the model
model = init_chat_model("groq:llama-3.3-70b-versatile")

# 3. Custom token counter (Consolidated to just one function)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

# 4. Create a message trimmer 
trimmer = trim_messages(
    max_tokens=200,          
    strategy="last",         
    token_counter=count_tokens,  
    include_system=True,     
    allow_partial=False      
)

# 5. Define the state modifier
def state_modifier(state):
    return trimmer.invoke(state["messages"])

# 6. Initialize Memory and Create the Agent
memory = MemorySaver()
agent = create_react_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=memory,
    prompt=state_modifier  # Updated to use 'prompt' for modern LangGraph
)

# 7. Config with corrected dictionary syntax
config = {"configurable": {"thread_id": "test-1"}}

# --- Example Usage ---
if __name__ == "__main__":
    query = "Please search for hotels in Tokyo."
    print(f"User: {query}")
    
    response = agent.invoke({"messages": [HumanMessage(content=query)]}, config)
    
    print("\nAgent:")
    print(response["messages"][-1].content)

C:\Users\hp\AppData\Local\Temp\ipykernel_25476\1034089803.py:41: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


User: Please search for hotels in Tokyo.

Agent:
Here are some hotels in Tokyo that you may be interested in:
1. Grand Hotel - a 5-star hotel with a spa, pool, and gym, priced at $350 per night.
2. City Inn - a 4-star hotel with a business center, priced at $180 per night.
3. Budget Stay - a 3-star hotel with free wifi, priced at $75 per night.


In [28]:
# Run Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Find hotels in {city}")
            ]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

Paris: ~1228 tokens, 40 messages
[HumanMessage(content='Please search for hotels in Tokyo.', additional_kwargs={}, response_metadata={}, id='6749ac2b-d02a-42f5-9b40-ef8018a0d4fa'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '33khnaaq3', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 229, 'total_tokens': 245, 'completion_time': 0.033751084, 'completion_tokens_details': None, 'prompt_time': 0.010906744, 'prompt_tokens_details': None, 'queue_time': 0.064032851, 'total_time': 0.044657828}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8603-762b-73e3-862e-73e078135b25-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Tokyo'}, 'id': '33khnaaq3', 'type': 'tool_call'}], invalid_tool_calls=

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kxjmwpwff4bvn01ja3azkcy4` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99498, Requested 595. Please try again in 1m20.352s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [38]:
import os
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

if __name__ == "__main__":
    # 1. Initialize model and memory
    model = init_chat_model("groq:llama-3.3-70b-versatile")
    memory = MemorySaver()

    # 2. Create the agent with a general interrupt on the tools node
    agent = create_react_agent(
        model=model,
        tools=[read_email_tool, send_email_tool],
        checkpointer=memory,
        interrupt_before=["tools"] # Pauses before ANY tool
    )

    config = {"configurable": {"thread_id": "email-thread-1"}}
    
    # Give the agent a multi-step task
    query = "Read email 123, then send an email to alice@test.com saying hi."
    print(f"User: {query}\n")
    
    # Initial run (this will pause as soon as it tries to use a tool)
    agent.invoke({"messages": [HumanMessage(content=query)]}, config)

    # 3. Handle the Human-in-the-Loop logic dynamically
    state = agent.get_state(config)
    
    # Loop while the agent is paused on a "next" node
    while state.next:
        # Get the tool calls the agent wants to execute
        last_message = state.values["messages"][-1]
        tool_names = [tc["name"] for tc in last_message.tool_calls]

        # Conditional Logic: Does it require human approval?
        if "send_email_tool" in tool_names:
            print(f"[ACTION REQUIRED] Agent wants to call: {tool_names}")
            print(f"Arguments: {last_message.tool_calls[0]['args']}")
            
            decision = input("Type 'approved' to continue or 'reject' to stop: ")
            
            if decision.lower() == "approved":
                # Passing None as the input resumes the graph from the paused state
                agent.invoke(None, config)
            else:
                print("Action rejected. Exiting process.")
                break # Exit the loop if rejected
        else:
            # If it's just read_email_tool, automatically approve and continue
            print(f"[AUTO-APPROVED] Agent calling: {tool_names}")
            agent.invoke(None, config)

        # Refresh the state to see if it paused again or finished
        state = agent.get_state(config)

    # 4. Print the final result once the graph finishes running
    if not state.next:
        print("\nFinal Agent Response:")
        print(agent.get_state(config).values["messages"][-1].content)

C:\Users\hp\AppData\Local\Temp\ipykernel_25476\1102481355.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


User: Read email 123, then send an email to alice@test.com saying hi.

[ACTION REQUIRED] Agent wants to call: ['read_email_tool', 'send_email_tool']
Arguments: {'email_id': '123'}

Final Agent Response:
Email content for ID: 123 
Email sent to alice@test.com with subject 'Hello'


In [39]:

config = {"configurable": {"thread_id": "test-approve"}}

# Step 1: Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to john@test.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

In [40]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='b4e1adf3-8d78-492a-92f1-9262c719bf5c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'edzrepfr0', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.046087181, 'completion_tokens_details': None, 'prompt_time': 0.017209013, 'prompt_tokens_details': None, 'queue_time': 0.160827971, 'total_time': 0.063296194}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8660-3389-75a1-87dc-8a3b03e3aa46-0', tool_calls=[{'name': 'send_email_tool', 'args': {'b

In [41]:
# Step 2: Approve
from langgraph.types import Command
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

In [42]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='b4e1adf3-8d78-492a-92f1-9262c719bf5c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'edzrepfr0', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.046087181, 'completion_tokens_details': None, 'prompt_time': 0.017209013, 'prompt_tokens_details': None, 'queue_time': 0.160827971, 'total_time': 0.063296194}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8660-3389-75a1-87dc-8a3b03e3aa46-0', tool_calls=[{'name': 'send_email_tool', 'args': {'b